<a href="https://colab.research.google.com/github/RohanYashraj/afdp-python-training/blob/main/2026%20AFDP%20Python%20Training/Notebook3_Agentic_AI_Agno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3 - A First Agent with Agno

**AFDP 2026 - Python Workshop (Block 3, about 20 minutes)**

Everything you did in Notebook 2 - filter, group, average - you can now hand to an **AI agent** and ask in plain English. This notebook builds one in about thirty lines using [Agno](https://docs.agno.com), an open-source Python framework for agents, and Google's Gemini model.

**What is an agent?** A language model on its own can only write text. An *agent* is a language model that has been given **tools** (ordinary Python functions), a set of **instructions**, and permission to loop: read the question, decide which tool to call, look at the result, decide whether it needs another, and finally answer. The arithmetic stays in your Python functions - auditable and deterministic. The model only decides *which* function to call and *how* to explain the result.

That division of labour is the whole point for actuaries: the numbers come from code you can check; the model handles the language.

## 1. Install the libraries

Colab does not have Agno installed, so the first cell installs it. This takes about 30 seconds. (The `-q` keeps the output quiet.)

In [90]:
!pip install -q agno google-genai
print("Installed.")

Installed.


## 2. Your Gemini API key

The agent needs a key to talk to Gemini. Google's free tier is enough for this workshop.

1. Go to [aistudio.google.com/apikey](https://aistudio.google.com/apikey) (sign in with your Google account) and click **Create API key**. Copy it.
2. In Colab, click the **key icon** in the left-hand sidebar (*Secrets*), click **Add new secret**, set the name to `GOOGLE_API_KEY`, paste the key as the value, and switch on **Notebook access**.
3. Run the cell below. It reads the secret without ever showing the key on screen.

If you are not in Colab, the cell will ask you to paste the key instead. Never type a key directly into a code cell - notebooks get shared, and keys in shared notebooks get misused.

In [91]:
import os

try:
    from google.colab import userdata                       # Colab only
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("Key loaded from Colab Secrets.")
except Exception:
    if not os.environ.get("GOOGLE_API_KEY"):
        from getpass import getpass
        os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")
    print("Key loaded.")

Key loaded from Colab Secrets.


## 3. Your first agent - custom instructions, no tools

Before we touch any data, build the simplest possible agent: a **model plus instructions**. The instructions are plain English sentences that turn a general-purpose model into a specialist - here, a tutor for trainee actuaries. There are no tools yet, so the agent can only draw on the model's general knowledge; it knows nothing about our dataset.

Run the cell, then read the instructions again and notice how they shape the tone and length of the answer.


In [92]:
from agno.agent import Agent
from agno.models.google import Gemini
from IPython.display import Markdown, display

MODEL_ID = "gemini-3.5-flash-lite"     # free-tier model; change here if Google renames it


def ask(agent: Agent, question: str) -> None:
    """Send a question to an agent and render the full reply as Markdown.

    (Streaming with print_response looks great in a terminal, but Colab
    clips the live-updating panel, so we display the finished answer instead.)"""
    response = agent.run(question)
    display(Markdown(response.content))


actuarial_tutor = Agent(
    model=Gemini(id=MODEL_ID),
    instructions=[
        "You are a friendly actuarial tutor helping trainee actuaries.",
        "Explain concepts in plain English, in at most four sentences.",
        "Where it helps, illustrate with a short health insurance example.",
    ],
    markdown=True,
)


In [93]:
ask(actuarial_tutor, "What is a loss ratio, and why do insurers watch it?")


A **loss ratio** is the percentage of premium dollars an insurer pays out in claims, calculated as total losses divided by earned premiums. For example, if a health insurer collects \$1,000 in premiums and pays out \$700 in medical bills, the loss ratio is 70%. Insurers watch this metric closely because it is the ultimate measure of their underwriting profitability. If the ratio creeps too high, it means claims are eating up all the cash needed for expenses and profit, signaling a need to raise rates.

Same agent, more questions. Run each cell and see how the instructions ("plain English", "at most four sentences", "health insurance example") shape every answer, whatever you ask.


In [94]:
ask(actuarial_tutor, "Explain adverse selection to a brand-new analyst.")


Adverse selection happens when people who are sickest or expect to use health services are more likely to buy insurance than healthy people. This happens because customers know more about their own health than the insurance company does. For example, someone who just found out they need knee surgery is much more eager to buy comprehensive coverage than someone who runs marathons daily. If insurers don't account for this, the pool of customers becomes unhealthier and more expensive than expected.

In [95]:
ask(actuarial_tutor, "Why do smokers usually pay higher health insurance premiums?")


Smokers typically pay higher health insurance premiums because they have a statistically higher risk of developing costly medical conditions like lung cancer and heart disease. Insurers charge more to cover this increased likelihood of frequent and expensive claims over their lifetime. For example, a 40-year-old smoker will likely face much higher monthly health premiums than a non-smoker of the same age to balance out the insurer's expected payout costs. This pricing strategy ensures fairness, so lower-risk policyholders aren't unfairly subsidizing the heavier healthcare costs of higher-risk individuals.

In [96]:
# Try changing the instructions above (for example: answer only in bullet points) and re-run
ask(actuarial_tutor, "What is the difference between a premium and a claim?")


Think of a **premium** as the regular price tag you pay to keep your health insurance active, while a **claim** is the bill you send to the insurer asking them to pay for your medical care. For instance, you pay a monthly premium of $200, and later file a $1,000 claim when you visit the doctor. In short, the premium is money flowing *in* from the policyholder, and the claim is money flowing *out* to cover healthcare costs.

## 4. Load the dataset

Same file, same loading cell as Notebook 2, so the agent works on data you already understand.

In [97]:
import os
import pandas as pd

FILE_NAME = "us_health_insurance_dataset_afdp.csv"
DATA_URL = "https://raw.githubusercontent.com/rohanyashraj/afdp-python-training/main/2026%20AFDP%20Python%20Training/us_health_insurance_dataset_afdp.csv"   # Direct link to the CSV in the course GitHub repo; set to "" to upload the file manually instead

if DATA_URL:
    insurance_data = pd.read_csv(DATA_URL)
elif os.path.exists(FILE_NAME):
    insurance_data = pd.read_csv(FILE_NAME)
else:
    try:
        from google.colab import files          # only exists inside Google Colab
        print("Please choose", FILE_NAME, "from your computer in the dialog below.")
        files.upload()
        insurance_data = pd.read_csv(FILE_NAME)
    except ImportError:
        raise FileNotFoundError(f"Put {FILE_NAME} in the same folder as this notebook and run this cell again.")

print("Loaded", len(insurance_data), "rows and", insurance_data.shape[1], "columns.")

Loaded 1338 rows and 7 columns.


## 5. A model without tools

First, ask Gemini a question about *our* data with no tools attached. Watch what happens: the model has never seen this file, so it can only guess, decline, or - worst of all - produce a confident number that is wrong. This is why tools matter.

In [98]:
plain_model = Agent(model=Gemini(id=MODEL_ID), markdown=True)   # no instructions, no tools


In [99]:
ask(plain_model, "In our health insurance dataset, what are the average annual charges for smokers in the southeast region?")


It looks like you forgot to include or attach the dataset! 

Please provide the dataset (or paste the relevant rows/data), and I will gladly calculate the average annual charges for smokers in the southeast region for you.

In [100]:
ask(plain_model, "How many policyholders in our dataset have a BMI above 30?")


It looks like you didn't include or attach your dataset! 

Please provide the dataset or upload the file, and I will be happy to count how many policyholders have a BMI above 30 for you.

In [101]:
ask(plain_model, "Which region in our dataset has the highest average annual charges?")


It looks like you forgot to include or attach the dataset! 

Please provide the dataset (or paste the relevant data/table here), and I will gladly analyze it to tell you which region has the highest average annual charges.

## 6. Give the agent tools

A tool is just a Python function with a clear name, typed inputs, and a docstring. The docstring is not decoration here - it is what the model reads to decide when to use the function, so write it as you would explain the function to a colleague.

Each tool below also prints a line when it runs, so you can see the agent working.

In [102]:
def average_charges(region: str = "all", smoker: str = "all") -> str:
    """Average annual charges in the health insurance dataset, optionally filtered.
    region: 'northeast', 'northwest', 'southeast', 'southwest' or 'all'.
    smoker: 'yes', 'no' or 'all'."""
    # print(f"   [tool called] average_charges(region={region!r}, smoker={smoker!r})")
    data = insurance_data
    if region != "all":
        data = data[data["region"] == region.lower()]
    if smoker != "all":
        data = data[data["smoker"] == smoker.lower()]
    if len(data) == 0:
        return "No policyholders match that filter."
    return f"{len(data)} policyholders, average annual charges {data['charges'].mean():,.2f}"


def large_claim_frequency(threshold: float = 20000) -> str:
    """Proportion of policyholders whose annual charges exceed a threshold, split by age band.
    Returns one line per age band with the number of lives and the frequency."""
    # print(f"   [tool called] large_claim_frequency(threshold={threshold})")
    bands = pd.cut(insurance_data["age"], bins=[17, 29, 39, 49, 59, 64],
                   labels=["18-29", "30-39", "40-49", "50-59", "60-64"])
    large = insurance_data["charges"] > threshold
    table = large.groupby(bands, observed=True).agg(["count", "mean"])
    lines = [f"{band}: {int(row['count'])} lives, frequency {row['mean']:.1%}" for band, row in table.iterrows()]
    return "\n".join(lines)


# Quick check that the tools work on their own - this is ordinary Python, no AI involved
print(average_charges("southeast", "yes"))
print(large_claim_frequency(20000))

   [tool called] average_charges(region='southeast', smoker='yes')
91 policyholders, average annual charges 34,845.00
   [tool called] large_claim_frequency(threshold=20000)
18-29: 417 lives, frequency 14.9%
30-39: 257 lives, frequency 17.9%
40-49: 279 lives, frequency 23.3%
50-59: 271 lives, frequency 24.0%
60-64: 114 lives, frequency 30.7%


In [103]:
agent = Agent(
    model=Gemini(id=MODEL_ID),
    tools=[average_charges, large_claim_frequency],
    instructions=[
        "You are an assistant to an actuary analysing a US health insurance dataset.",
        "Always use the tools to get numbers. Never estimate or invent figures.",
        "Quote the numbers the tools return, then add one or two sentences of interpretation.",
    ],
    markdown=True,
)


In [104]:
ask(agent, "What are the average annual charges for smokers in the southeast region, and how does that compare with non-smokers there?")


   [tool called] average_charges(region='southeast', smoker='yes')
   [tool called] average_charges(region='southeast', smoker='no')


The average annual charges for smokers in the southeast region are **$34,845.00** (across 91 policyholders), while non-smokers in the same region average **$8,032.22** (across 273 policyholders). 

This represents a substantial difference, with smokers in the southeast incurring average annual costs that are more than four times higher than those of non-smokers.

In [105]:
ask(agent, "How many policyholders are in the northwest region, and what are their average annual charges?")


   [tool called] average_charges(region='northwest', smoker='all')


There are **325** policyholders in the northwest region, and their average annual charges are **$12,417.58**. 

This gives a baseline cost expectation for the northwest territory, which can be further segmented by risk factors like smoking status or age for more granular pricing analysis.

In [106]:
ask(agent, "Do smokers in the northeast pay more on average than smokers in the southwest?")


   [tool called] average_charges(region='northeast', smoker='yes')
   [tool called] average_charges(region='southwest', smoker='yes')


- Smokers in the northeast: $29,673.54 (67 policyholders)
- Smokers in the southwest: $32,269.06 (58 policyholders)

Contrary to what one might expect based on regional pricing alone, smokers in the southwest actually pay more on average than smokers in the northeast by about $2,595.52. This suggests that other rating factors within the southwest smoker cohort (such as age or BMI distributions) may be driving up their average costs.

**What happened.** The model read the question, worked out that it needed `average_charges` twice (once for smokers, once for non-smokers), called your function, and wrote the comparison. The numbers are pandas' numbers - the same ones you would get from Notebook 2 - and the sentences are the model's.

Now a question that needs *both* tools and a little reasoning.

In [107]:
ask(
    agent,
    "Which age band has the highest frequency of large claims above 25,000? "
    "Then tell me whether smoking or age looks like the bigger driver of cost, using the tools.",
)


   [tool called] large_claim_frequency(threshold=25000.0)
   [tool called] average_charges(region='all', smoker='all')
   [tool called] average_charges(region='all', smoker='yes')
   [tool called] average_charges(region='all', smoker='no')
   [tool called] average_charges(region='northeast', smoker='all')


Based on the health insurance dataset, here are the breakdowns for large claims and cost drivers:

### Large Claim Frequency (> $25,000)
- **18-29:** 12.0% (417 lives)
- **30-39:** 12.1% (257 lives)
- **40-49:** 14.7% (279 lives)
- **50-59:** 16.6% (271 lives)
- **60-64:** 29.8% (114 lives)

The **60–64** age band has the highest frequency of large claims at 29.8%, more than double the rate of younger cohorts.

### Cost Drivers: Smoking vs. Age
- **Overall average charges:** $13,270.42
- **Non-smoker average charges:** $8,434.27
- **Smoker average charges:** $32,050.23

Smoking appears to be a much larger driver of cost than age; smokers incur average annual charges nearly four times higher than non-smokers ($32,050.23 compared to $8,434.27).

In [108]:
ask(agent, "How does the frequency of claims above 40,000 vary by age band?")


   [tool called] large_claim_frequency(threshold=40000.0)


Here is the frequency of claims exceeding $40,000 by age band:

* **18–29:** 1.2% (out of 417 lives)
* **30–39:** 3.9% (out of 257 lives)
* **40–49:** 8.2% (out of 279 lives)
* **50–59:** 8.9% (out of 271 lives)
* **60–64:** 14.9% (out of 114 lives)

The frequency of large claims ($>$\$40,000) increases steadily with age, jumping sharply from 1.2% in the youngest bracket to 14.9% among policyholders aged 60–64. This upward trend highlights the growing risk and cost severity associated with older age demographics in the health insurance portfolio.

In [109]:
ask(
    agent,
    "Compare average charges for smokers and non-smokers across all four regions, "
    "and tell me which region shows the largest smoking gap.",
)


   [tool called] average_charges(region='all', smoker='yes')
   [tool called] average_charges(region='all', smoker='no')
   [tool called] average_charges(region='northeast', smoker='yes')
   [tool called] average_charges(region='northeast', smoker='no')
   [tool called] average_charges(region='northwest', smoker='yes')
   [tool called] average_charges(region='northwest', smoker='no')
   [tool called] average_charges(region='southeast', smoker='yes')
   [tool called] average_charges(region='southeast', smoker='no')
   [tool called] average_charges(region='southwest', smoker='yes')
   [tool called] average_charges(region='southwest', smoker='no')


Here is the comparison of average annual charges for smokers and non-smokers across all four regions:

| Region | Smoker Average Charges ($) | Non-Smoker Average Charges ($) | Difference (Smoking Gap) ($) |
| :--- | :--- | :--- | :--- |
| **Northeast** | 29,673.54 | 9,165.53 | 20,508.01 |
| **Northwest** | 30,192.00 | 8,556.46 | 21,635.54 |
| **Southeast** | 34,845.00 | 8,032.22 | 26,812.78 |
| **Southwest** | 32,269.06 | 8,019.28 | 24,249.78 |

The **Southeast** region shows the largest smoking gap, with smokers incurring an average of $26,812.78 more per year than non-smokers. Across all regions, smoking status introduces a massive cost differential, roughly quadrupling average annual charges.

## 7. What to take away

- An agent is **model + tools + instructions**, running in a loop. Nothing more mysterious than that.
- The **tools are the actuarial content.** Writing a good tool is writing a good, well-documented Python function - which is what Notebooks 1 and 2 were about.
- The instructions are your **guardrails**: "never invent figures" is a policy, and you wrote it.
- Everything you would want to audit - the filters, the thresholds, the arithmetic - lives in your code, not inside the model.

**Where this goes next.** Give the agent a tool that runs a GLM, one that reads a policy PDF, one that writes an Excel report. Give a *team* of agents different roles (data checker, modeller, reviewer). Same pattern, bigger tools.

## Your turn (if there is time)

1. Add a third tool, `average_bmi(region)`, and ask the agent a question that needs it.
2. Change the instructions so the agent always answers in exactly three bullet points, and see whether it obeys.
3. Ask a question the tools *cannot* answer (for example about motor insurance) and check that the agent says so instead of making something up.

In [110]:
# Try your own code here